In [1]:
from the_well.data.normalization import ZScoreNormalization
from matplotlib import pyplot as plt
from pathlib import Path
import numpy as np
import pandas as pd
import shutil
import pickle
import fsspec
import torch
import yaml
import sys
import re

PROJECT_ROOT = "/Users/gabrieleaccarino/Workspace/waveletflux"
sys.path.insert(0, PROJECT_ROOT)

from waveletflux.utils.well_data import get_well_dataset, get_well_inputs
from waveletflux.modules.wavelet_fm import WaveletFlowModule
# your real spectral helpers — used verbatim by the metrics in Section 4
from waveletflux.utils.fourier import (
    isotropic_power_spectrum, isotropic_cross_correlation,
)

rc = {
    "axes.axisbelow": True, "axes.linewidth": 1.0,
    "axes.labelsize": "x-small", "axes.titlesize": "x-small",
    "figure.dpi": 150, "figure.figsize": (3.4, 2.6), "font.size": 12.0,
    "legend.fontsize": "xx-small", "lines.linewidth": 1.2,
    "savefig.transparent": True,
    "xtick.labelsize": "xx-small", "xtick.major.width": 1.0,
    "ytick.labelsize": "xx-small", "ytick.major.width": 1.0,
    "font.family": ["serif"],
    "text.usetex": shutil.which("latex") is not None,
    "text.latex.preamble": r"\usepackage{amsfonts}",
}
plt.rcParams.update(rc)

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print("Imports OK | device:", device)

Imports OK | device: mps


In [8]:
NOISE_LEVELS = ['0', '1', '01', '001']
SEEDS = [42, 84, 168]
well_dataset_name = 'turbulent_radiative_layer_2D'

# Rollout-horizon windows for TRL, matching Table S11 (1-indexed, inclusive).
HORIZONS = [("01:32", 1, 32), ("33:64", 33, 64), ("65:97", 65, 97)]

In [6]:
dfs_metrics = []
for noise_level in NOISE_LEVELS:
    for seed in SEEDS:
        metrics_csv_path = (f'{PROJECT_ROOT}/experiments/{well_dataset_name}'
                            f'/metrics/WaveletFlow_{well_dataset_name}_3scales_haar_v4.0_'
                            f'nsampling50_nens8_seed{seed}_ic0p{noise_level}_metrics.csv')
        df = pd.read_csv(metrics_csv_path)
        dfs_metrics.append(df)

metrics_df = pd.concat(dfs_metrics, ignore_index=True)

In [23]:
import pandas as pd
import numpy as np

df = metrics_df

# per-field ML rows only (drop global-diagnostic rows if present)
d = df[df["field"] != "_global_"].copy() if "_global_" in df["field"].unique() else df.copy()

HORIZONS = [("01:32", 0, 31), ("33:64", 32, 63), ("65:97", 64, 96)]

def seed_agg(frame, col, lo=None, hi=None):
    """mean over (field, step-in-window, traj) per (ic_noise, seed),
       then mean/std across seeds -> DataFrame indexed by ic_noise."""
    sub = frame
    if lo is not None:
        sub = sub[(sub["step"] >= lo) & (sub["step"] <= hi)]
    per_seed = sub.groupby(["ic_noise", "seed"])[col].mean().reset_index()
    return per_seed.groupby("ic_noise")[col].agg(["mean", "std"])

def fmt(agg):
    return agg.apply(lambda r: f"{r['mean']:.3f} ± {0.0 if pd.isna(r['std']) else r['std']:.3f}", axis=1)

# ---- CRPS across horizons ----
crps_tbl = pd.DataFrame({
    "CRPS 01:32": fmt(seed_agg(d, "crps", 0, 31)),
    "CRPS 33:64": fmt(seed_agg(d, "crps", 32, 63)),
    "CRPS 65:97": fmt(seed_agg(d, "crps", 64, 96)),
    #"CRPS all":   fmt(seed_agg(d, "crps")),
})
crps_tbl.index.name = "ic_noise"

# ---- Spectral coherence across bands (all steps) ----
spec_tbl = pd.DataFrame({
    "SpecCoh Low":   fmt(seed_agg(d, "rmse_c_low")),
    "SpecCoh Mid":   fmt(seed_agg(d, "rmse_c_mid")),
    "SpecCoh High":  fmt(seed_agg(d, "rmse_c_high")),
    #"SpecCoh vHigh": fmt(seed_agg(d, "rmse_c_vhigh")),
})
spec_tbl.index.name = "ic_noise"

print("CRPS across rollout horizons (mean ± std over seeds)\n", crps_tbl, "\n")
print("Spectral coherence RMSE across bands (mean ± std over seeds)\n", spec_tbl)

# LaTeX for the paper
print(crps_tbl.to_latex())
print(spec_tbl.to_latex())

CRPS across rollout horizons (mean ± std over seeds)
              CRPS 01:32     CRPS 33:64     CRPS 65:97
ic_noise                                             
0.000     0.410 ± 0.003  1.018 ± 0.011  1.178 ± 0.030
0.001     0.410 ± 0.003  1.036 ± 0.016  1.180 ± 0.024
0.010     0.410 ± 0.003  1.030 ± 0.016  1.190 ± 0.007
0.100     0.413 ± 0.003  1.026 ± 0.040  1.203 ± 0.012 

Spectral coherence RMSE across bands (mean ± std over seeds)
             SpecCoh Low    SpecCoh Mid   SpecCoh High
ic_noise                                             
0.000     0.133 ± 0.001  0.174 ± 0.000  0.141 ± 0.000
0.001     0.133 ± 0.001  0.174 ± 0.001  0.141 ± 0.000
0.010     0.133 ± 0.001  0.174 ± 0.001  0.141 ± 0.001
0.100     0.133 ± 0.000  0.174 ± 0.000  0.141 ± 0.001
\begin{tabular}{llll}
\toprule
 & CRPS 01:32 & CRPS 33:64 & CRPS 65:97 \\
ic_noise &  &  &  \\
\midrule
0.000000 & 0.410 ± 0.003 & 1.018 ± 0.011 & 1.178 ± 0.030 \\
0.001000 & 0.410 ± 0.003 & 1.036 ± 0.016 & 1.180 ± 0.024 \\
0.010000 &

In [15]:
crps_tbl

,CRPS 01:32,CRPS 33:64,CRPS 65:97,CRPS all
ic_noise,,,,
0.001,0.407 ± 0.000,1.054 ± 0.000,1.160 ± 0.000,0.877 ± 0.000


In [10]:
def agg_over_seeds(df, value_col, step_lo=None, step_hi=None):
    '''mean over (field,step[in range],traj) per (ic_noise,seed), then mean/std over seeds.'''
    sub = df
    if step_lo is not None:
        sub = sub[(sub.step >= step_lo) & (sub.step <= step_hi)]
    per_seed = sub.groupby(["ic_noise", "seed"])[value_col].mean().reset_index()
    return per_seed.groupby("ic_noise")[value_col].agg(["mean", "std"]).reset_index()

def cell(agg, ic_noise):
    r = agg[agg.ic_noise == ic_noise].iloc[0]
    s = 0.0 if np.isnan(r["std"]) else r["std"]
    return f"{r['mean']:.3f} ± {s:.3f}"

rows = []
for ic_noise in NOISE_LEVELS:
    row = {"ic_noise": ic_noise}
    for name, lo, hi in HORIZONS:
        row[f"VRMSE {name}"] = cell(agg_over_seeds(df, "vrmse", lo, hi), ic_noise)
    row["VRMSE all"] = cell(agg_over_seeds(df, "vrmse"), ic_noise)
    row["CRPS all"]  = cell(agg_over_seeds(df, "crps"), ic_noise)
    for band, lab in [("low", "Low"), ("mid", "Mid"), ("high", "High")]:
        row[f"SpecCoh {lab}"] = cell(agg_over_seeds(df, f"rmse_c_{band}"), ic_noise)
    rows.append(row)

summary = pd.DataFrame(rows).set_index("ic_noise")
#summary.to_csv(results_dir / "ic_sensitivity_summary.csv")
#print("saved:", (results_dir / 'ic_sensitivity_summary.csv').name)
summary

IndexError: single positional indexer is out-of-bounds

In [ ]:
# ============================================================
# Main appendix tables
# ============================================================

def build_main_table(
    metric: str,
    models: list = None,
    pdes: list = None,
    raw_horizons: bool = False,
) -> pd.DataFrame:
    # Build a unified multi-PDE appendix table.
    #
    # Parameters
    # ----------
    # metric        : stats_df column to aggregate (e.g. "vrmse", "crps").
    # models        : ordered model keys; defaults to MODEL_ORDER.
    # pdes          : ordered PDE keys;   defaults to PDEs.
    # raw_horizons  : if False (default), columns are Short/Mid/Long and the
    #                 index is a (PDE_label, model_label) MultiIndex, suitable
    #                 for Jupyter display.
    #                 if True, columns are a (pde_key, raw_horizon_str) MultiIndex
    #                 and the index is flat model display labels -- the layout
    #                 expected by to_latex_wide().
    if models is None: models = MODEL_ORDER
    if pdes   is None: pdes   = PDEs

    frames = []
    for pde in pdes:
        sub = (
            stats_df[(stats_df["pde"] == pde) & (stats_df["model"].isin(models))]
            [["model", "horizon", metric]]
            .groupby(["model", "horizon"])[metric]
            .agg(aggregate).round(3).reset_index()
        )
        if raw_horizons:
            # Keep raw horizon strings; produce (pde_key, hz) MultiIndex columns
            sub = sub.pivot(index="model", columns="horizon", values=metric)
            raw_order = [h for h in HORIZON_BINS_RAW[pde] if h in sub.columns]
            sub = sub.reindex(
                index=[m for m in models if m in sub.index],
                columns=raw_order,
            )
            sub.index   = [labels.get(m, m) for m in sub.index]
            sub.columns = pd.MultiIndex.from_tuples(
                [(pde, h) for h in sub.columns], names=["pde", "horizon"]
            )
        else:
            # Map to generic Short/Mid/Long; produce (PDE_label, model) MultiIndex rows
            sub["horizon"] = sub["horizon"].map(HORIZON_TO_GENERIC[pde])
            sub = sub.pivot(index="model", columns="horizon", values=metric)
            sub = sub.reindex(
                index=[m for m in models if m in sub.index],
                columns=[h for h in GENERIC_HORIZONS if h in sub.columns],
            )
            sub.index = pd.MultiIndex.from_tuples(
                [(PDE_LABELS[pde], labels.get(m, m)) for m in sub.index],
                names=["PDE", "Model"],
            )
        frames.append(sub)

    result = pd.concat(frames, axis=1 if raw_horizons else 0)
    if raw_horizons:
        result.index.name = "Model"
    return result


def build_spectral_table(
    models: list = None,
    pdes: list = None,
) -> pd.DataFrame:
    # Multi-PDE spectral coherence RMSE table (averaged over horizons).
    # Rows : (PDE, model) MultiIndex
    # Cols : Low / Mid / High
    if models is None: models = MODEL_ORDER
    if pdes   is None: pdes   = PDEs

    frames = []
    for pde in pdes:
        sub = (
            stats_df[(stats_df["pde"] == pde) & (stats_df["model"].isin(models))]
            [["model", "horizon"] + list(BAND_COLS.keys())]
            .groupby(["model", "horizon"])[list(BAND_COLS.keys())]
            .agg(aggregate).reset_index()
            .rename(columns=BAND_COLS)
            .groupby("model")[BANDS]
            .mean().round(3)
        )
        sub = sub.reindex([m for m in models if m in sub.index])
        sub.index = pd.MultiIndex.from_tuples(
            [(PDE_LABELS[pde], labels.get(m, m)) for m in sub.index],
            names=["PDE", "Model"],
        )
        frames.append(sub)
    return pd.concat(frames)


def build_spectral_horizon_table(
    models: list = None,
    pdes: list = None,
) -> pd.DataFrame:
    # Per-PDE spectral table broken down by horizon AND band.
    # Rows : (PDE, model) MultiIndex
    # Cols : (band, horizon) MultiIndex
    if models is None: models = MODEL_ORDER
    if pdes   is None: pdes   = PDEs

    frames = []
    for pde in pdes:
        sub = (
            stats_df[(stats_df["pde"] == pde) & (stats_df["model"].isin(models))]
            [["model", "horizon"] + list(BAND_COLS.keys())]
            .melt(id_vars=["model", "horizon"],
                  value_vars=list(BAND_COLS.keys()),
                  var_name="band_key", value_name="srmse")
            .assign(band=lambda x: x["band_key"].map(BAND_COLS),
                    horizon=lambda x: x["horizon"].map(HORIZON_TO_GENERIC[pde]))
            .groupby(["model", "band", "horizon"])["srmse"]
            .agg(aggregate).round(3).reset_index()
            .pivot(index="model", columns=["band", "horizon"], values="srmse")
            .reindex(columns=BANDS, level="band")
            .reindex(columns=GENERIC_HORIZONS, level="horizon")
        )
        sub = sub.reindex([m for m in models if m in sub.index])
        sub.index = pd.MultiIndex.from_tuples(
            [(PDE_LABELS[pde], labels.get(m, m)) for m in sub.index],
            names=["PDE", "Model"],
        )
        frames.append(sub)
    return pd.concat(frames)


def build_w1_table(
    models: list = None,
    pdes: list = None,
) -> pd.DataFrame:
    # Wasserstein-1 table: (PDE, model) rows x single W1 column.
    if models is None: models = WFM_all + gen_models
    if pdes   is None: pdes   = PDEs

    frames = []
    for pde in pdes:
        sub = (
            stats_df[(stats_df["pde"] == pde) & (stats_df["model"].isin(models))]
            [["model", "w1"]]
            .groupby("model")["w1"]
            .agg(aggregate).round(3)
            .reset_index().set_index("model")
        )
        sub = sub.reindex([m for m in models if m in sub.index])
        sub.index = pd.MultiIndex.from_tuples(
            [(PDE_LABELS[pde], labels.get(m, m)) for m in sub.index],
            names=["PDE", "Model"],
        )
        sub.columns = [r"$W_1$"]
        frames.append(sub)
    return pd.concat(frames)


# ============================================================
# Ablation tables
# ============================================================

def build_ablation_table(
    metric: str,
    pde: str,
    param: str,
    models: list = None,
    param_filter=None,
) -> pd.DataFrame:
    # Ablation table: model rows x param-value columns.
    #
    # metric       : "vrmse", "crps", etc.
    # pde          : PDE key string.
    # param        : physical parameter column (e.g. "tcool", "Rayleigh", "alpha").
    # models       : models to show; defaults to WFM_all + gen_models.
    # param_filter : optional callable applied to param values,
    #                e.g. ``lambda v: v <= 0.03`` to restrict tcool range.
    if models is None: models = WFM_all + gen_models

    sub = (
        stats_df[(stats_df["pde"] == pde) & (stats_df["model"].isin(models))]
        [["model", param, metric]]
        .dropna(subset=[param, metric])
    )
    if param_filter is not None:
        sub = sub[sub[param].apply(param_filter)]

    sub = (
        sub.groupby(["model", param])[metric]
        .agg(aggregate).round(3).reset_index()
        .pivot(index="model", columns=param, values=metric)
    )
    sub = sub.reindex([m for m in models if m in sub.index])
    sub.index = [labels.get(m, m) for m in sub.index]
    sub.index.name  = "Model"
    sub.columns.name = param
    return sub


def build_spectral_ablation_table(
    pde: str,
    param: str,
    models: list = None,
    param_filter=None,
) -> pd.DataFrame:
    # Spectral ablation table: model rows x (band, param_value) columns.
    if models is None: models = WFM_all + gen_models

    sub = (
        stats_df[(stats_df["pde"] == pde) & (stats_df["model"].isin(models))]
        [["model", param] + list(BAND_COLS.keys())]
        .dropna(subset=[param])
    )
    if param_filter is not None:
        sub = sub[sub[param].apply(param_filter)]

    sub = (
        sub.melt(id_vars=["model", param],
                 value_vars=list(BAND_COLS.keys()),
                 var_name="band_key", value_name="srmse")
        .assign(band=lambda x: x["band_key"].map(BAND_COLS))
        .groupby(["model", "band", param])["srmse"]
        .agg(aggregate).round(3).reset_index()
        .pivot(index="model", columns=["band", param], values="srmse")
        .reindex(columns=BANDS, level="band")
    )
    sub = sub.reindex([m for m in models if m in sub.index])
    sub.index = [labels.get(m, m) for m in sub.index]
    sub.index.name = "Model"
    return sub